# SAGE CIFAR-10 Colab

Use this notebook to run the CIFAR-10 SAGE channel-pruning experiments on a Colab GPU.

Before running this notebook, make sure the latest local code is pushed to GitHub. Colab clones from the remote repo, so uncommitted local files are invisible there.

## 1. Check GPU

In Colab, choose `Runtime -> Change runtime type -> T4 GPU` or better before running the notebook.

In [ ]:
!nvidia-smi

import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0))

## 2. Clone And Install

This pulls `main` from the GitHub remote. If the repo is private, Colab will ask for authentication or you can upload a zip manually.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/SelbinyyazS/superweights--dynamic.git"
BRANCH = "main"
WORKDIR = "/content/superweights--dynamic"

def clone_url_with_optional_token(repo_url):
    """Use a Colab Secret named GITHUB_TOKEN for private repos, if present."""
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    if token and repo_url.startswith("https://github.com/"):
        return repo_url.replace("https://github.com/", f"https://{token}@github.com/", 1)
    return repo_url

if not os.path.exists(WORKDIR):
    result = subprocess.run(
        ["git", "clone", "--branch", BRANCH, clone_url_with_optional_token(REPO_URL), WORKDIR],
        text=True,
        capture_output=True,
    )
    if result.returncode != 0:
        print(result.stderr.replace(clone_url_with_optional_token(REPO_URL), REPO_URL))
        raise RuntimeError(
            "Git clone failed. If this is a private repo, add a Colab Secret named "
            "GITHUB_TOKEN with repo read access, then rerun this cell."
        )

os.chdir(WORKDIR)
print("cwd", os.getcwd())
!git fetch origin {BRANCH}
!git checkout {BRANCH}
!git pull origin {BRANCH}
!pip install -q -r requirements.txt

## 3. Synthetic Smoke Test

This does not download CIFAR-10. It checks masking, SAGE scores, channel pruning, and compaction.

In [ ]:
!python smoke_test.py

## 4. Short CIFAR-10 GPU Sanity Run

This downloads CIFAR-10 and runs only a small number of batches. Run this before starting the long sweep.

In [ ]:
!python train.py \
  --dataset cifar10 \
  --model cifar_cnn \
  --dense_start \
  --epochs 1 \
  --base_channels 32 \
  --batch_size 128 \
  --train_batches 50 \
  --eval_batches 20 \
  --num_workers 2 \
  --log_path logs/colab_cifar10_sanity.csv

## 5. Full CIFAR-10 SAGE Comparison

This runs dense, SAGE, magnitude, and random pruning. The defaults are a Colab-friendly starter sweep. For the stronger experiment, use `SEEDS = [1, 2, 3]`, `EPOCHS = 80`, `POST_COMPACT_EPOCHS = 20`, and `BASE_CHANNELS = 64`.

In [ ]:
import subprocess

SEEDS = [1]
EPOCHS = 40
POST_COMPACT_EPOCHS = 10
BASE_CHANNELS = 48
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OUTPUT_DIR = "/content/drive/MyDrive/sage/cifar10_colab_stage3" if SAVE_TO_DRIVE else "logs/cifar10_colab_stage3"
AGGREGATE_CSV = "/content/drive/MyDrive/sage/cifar10_colab_stage3_aggregate.csv" if SAVE_TO_DRIVE else "results/cifar10_colab_stage3_aggregate.csv"

cmd = [
    "python", "run_multiseed.py",
    "--dataset", "cifar10",
    "--model", "cifar_cnn",
    "--seeds", *map(str, SEEDS),
    "--epochs", str(EPOCHS),
    "--post_compact_epochs", str(POST_COMPACT_EPOCHS),
    "--base_channels", str(BASE_CHANNELS),
    "--output_dir", OUTPUT_DIR,
    "--neuron_prune_start_epoch", "20",
    "--neuron_prune_end_epoch", str(EPOCHS),
    "--neuron_prune_interval", "2",
    "--neuron_prune_fraction", "0.03",
    "--sage_focus_start_epoch", "20",
    "--sage_grad_boost", "1.15",
    "--sage_boost_fraction", "0.03",
    "--weak_grad_decay", "0.95",
    "--num_workers", "2",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True)

## 6. Summarize Results

In [ ]:
import glob
import subprocess

OUTPUT_DIR = globals().get("OUTPUT_DIR", "logs/cifar10_colab_stage3")
AGGREGATE_CSV = globals().get("AGGREGATE_CSV", "results/cifar10_colab_stage3_aggregate.csv")
logs = sorted(glob.glob(f"{OUTPUT_DIR}/*.csv"))
print("logs", len(logs))
subprocess.run(["python", "summarize_logs.py", "--aggregate", *logs, "--output", AGGREGATE_CSV], check=True)
print(open(AGGREGATE_CSV).read())

## 7. Download Logs

This zips the run logs and aggregate CSV so you can bring them back to the local repo.

In [ ]:
OUTPUT_DIR = globals().get("OUTPUT_DIR", "logs/cifar10_colab_stage3")
AGGREGATE_CSV = globals().get("AGGREGATE_CSV", "results/cifar10_colab_stage3_aggregate.csv")
!zip -r sage_cifar10_colab_results.zip {OUTPUT_DIR} {AGGREGATE_CSV}

from google.colab import files

files.download("sage_cifar10_colab_results.zip")